# Cleaned and Functionalized Ephys Analysis Notebook
This notebook contains a functionalized and organized version of the original analysis, with reusable functions and clear structure for reproducible analysis and visualization.

In [ ]:
import scipy
import numpy as np
import matplotlib.pyplot as plt
import os
from matplotlib import cm

# --- Data Loading Functions ---
def load_mat_file(mat_path):
    data = scipy.io.loadmat(mat_path, struct_as_record=False, squeeze_me=True)
    return data

def mat_struct_to_dict(obj):
    if isinstance(obj, np.ndarray):
        if obj.dtype == 'O':
            return [mat_struct_to_dict(o) for o in obj]
        else:
            return obj
    elif hasattr(obj, '_fieldnames'):
        return {field: mat_struct_to_dict(getattr(obj, field)) for field in obj._fieldnames}
    else:
        return obj

def get_session_dict(data_dict1):
    try:
        data_dict = mat_struct_to_dict(data_dict1['data'])
    except KeyError:
        data_dict = mat_struct_to_dict(data_dict1['ans'])
    return data_dict

def get_drilled_down(data_dict, session_key, subkey):
    return data_dict[session_key][subkey]

# --- Analysis Functions ---
def get_trials_and_outcomes(session):
    behav_data = session['behav_data']
    trials = behav_data['trials_data_exp']
    outcomes = np.array([trial['trialoutcome'] for trial in trials])
    return trials, outcomes

def get_event_times(ni_events, align_to, trials, outcomes, outcome_of_interest=None):
    n_trials = len(trials)
    if align_to in ni_events:
        event_arr = ni_events[align_to]
        if isinstance(event_arr, dict) and 'rise_t' in event_arr:
            event_times_all = np.array(event_arr['rise_t']).flatten()
        else:
            event_times_all = np.array(event_arr).flatten()
        if outcome_of_interest is not None:
            mask = outcomes == outcome_of_interest
            event_times = event_times_all[mask]
            trial_indices = np.where(mask)[0]
            trial_outcomes = outcomes[mask]
        else:
            event_times = event_times_all
            trial_indices = np.arange(n_trials)
            trial_outcomes = outcomes
        align_label = align_to
    elif align_to == 'trialoutcome_time':
        event_times_all = np.array([trial['hit_time'] for trial in trials])
        if outcome_of_interest is not None:
            mask = outcomes == outcome_of_interest
            event_times = event_times_all[mask]
            trial_indices = np.where(mask)[0]
            trial_outcomes = outcomes[mask]
        else:
            event_times = event_times_all
            trial_indices = np.arange(n_trials)
            trial_outcomes = outcomes
        align_label = 'Hit Time'
    else:
        raise ValueError(f"Unknown align_to: {align_to}")
    return event_times, trial_indices, align_label, trial_outcomes

# --- Visualization Functions ---
def plot_psth_raster_grid_sorted(session, align_to='Change_ON', outcome_of_interest='Hit', clusters_per_row=10, min_trials_with_spikes=20, save_figure=False):
    npx_probes = session['NPX_probes']
    ni_events = session['NI_events']
    trials, outcomes = get_trials_and_outcomes(session)
    spike_times = npx_probes['st']
    cluster_ids = npx_probes['clu']
    event_times, trial_indices, align_label, trial_outcomes = get_event_times(ni_events, align_to, trials, outcomes, outcome_of_interest)
    good_clusters = npx_probes.get('cluster_id_KS_good', np.unique(cluster_ids))
    window = [-0.5, 1.0]
    bin_size = 0.05
    bins = np.arange(window[0], window[1] + bin_size, bin_size)
    window_length = window[1] - window[0]
    clusters_to_plot = []
    rasters_for_plot = []
    counts_for_plot = []
    mean_rates = []
    for clu in good_clusters:
        spike_times_clu = spike_times[cluster_ids == clu]
        all_counts = []
        all_rasters = []
        for t in event_times:
            aligned = spike_times_clu - t
            mask = (aligned >= window[0]) & (aligned <= window[1])
            all_rasters.append(aligned[mask])
            all_counts.append(np.histogram(aligned[mask], bins=bins)[0])
        n_trials_with_spikes = sum(len(ras) > 0 for ras in all_rasters)
        if n_trials_with_spikes > min_trials_with_spikes:
            clusters_to_plot.append(clu)
            rasters_for_plot.append(all_rasters)
            counts_for_plot.append(np.array(all_counts))
            mean_rate = np.sum(all_counts) / (len(all_counts) * window_length)
            mean_rates.append(mean_rate)
    if len(mean_rates) == 0:
        print("No clusters passed the spike-in-trials filter. Adjust your parameters.")
        return
    sort_idx = np.argsort(mean_rates)[::-1]
    clusters_to_plot = [clusters_to_plot[i] for i in sort_idx]
    rasters_for_plot = [rasters_for_plot[i] for i in sort_idx]
    counts_for_plot = [counts_for_plot[i] for i in sort_idx]
    mean_rates = [mean_rates[i] for i in sort_idx]
    n_plot = len(clusters_to_plot)
    n_cols = clusters_per_row * 2 + (clusters_per_row - 1)
    n_rows = int(np.ceil(n_plot / clusters_per_row))
    width_ratios = []
    for i in range(clusters_per_row):
        width_ratios.extend([1, 1])
        if i < clusters_per_row - 1:
            width_ratios.append(0.3)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(2.2 * n_cols, 1.8 * n_rows), sharex='col', gridspec_kw={'width_ratios': width_ratios})
    if n_rows == 1:
        axes = np.expand_dims(axes, axis=0)
    for idx, (clu, all_rasters, all_counts) in enumerate(zip(clusters_to_plot, rasters_for_plot, counts_for_plot)):
        row = idx // clusters_per_row
        col_pair = idx % clusters_per_row
        base_col = col_pair * 3
        ax_psth = axes[row, base_col]
        ax_raster = axes[row, base_col + 1]
        psth = np.mean(all_counts, axis=0) / bin_size if len(all_counts) > 0 else np.zeros(len(bins)-1)
        ax_psth.step(bins[:-1], psth, where='post', color='k')
        ax_psth.axvline(0, color='r', linestyle='--', lw=0.8)
        ax_psth.set_ylim(bottom=0)
        ax_psth.tick_params(axis='y', labelsize=7)
        ax_psth.text(0.98, 0.95, f'Clu {clu}', ha='right', va='top', fontsize=8, transform=ax_psth.transAxes,
                     bbox=dict(facecolor='white', edgecolor='gray', boxstyle='round,pad=0.2', alpha=0.7))
        if row == n_rows - 1:
            xticks = [window[0], -0.5, 0, 0.5, window[1]]
            xticks = sorted(set([x for x in xticks if window[0] <= x <= window[1]]))
            ax_psth.set_xticks(xticks)
            ax_psth.set_xticklabels([f"{b:.2f}" for b in xticks], rotation=0, fontsize=9)
            ax_psth.set_xlabel(f'Time from {align_label} (s)')
        else:
            ax_psth.set_xticklabels([])
        if col_pair == 0:
            ax_psth.set_ylabel('Firing rate (Hz)')
        n_trials_raster = len(all_rasters)
        for trial, ras in enumerate(all_rasters):
            ax_raster.vlines(ras, trial + 0.5, trial + 1.5, color='k', lw=0.5)
        ax_raster.axvline(0, color='r', linestyle='--', lw=0.8)
        ax_raster.set_ylim(0.5, n_trials_raster + 0.5)
        if n_trials_raster >= 3:
            trial_ticks = [1, n_trials_raster // 2 + 1, n_trials_raster]
        else:
            trial_ticks = list(range(1, n_trials_raster + 1))
        ax_raster.set_yticks(trial_ticks)
        ax_raster.set_yticklabels([str(t) for t in trial_ticks], fontsize=7)
        ax_raster.invert_yaxis()
        ax_raster.text(0.98, 0.95, f'Clu {clu}', ha='right', va='top', fontsize=8, transform=ax_raster.transAxes,
                      bbox=dict(facecolor='white', edgecolor='gray', boxstyle='round,pad=0.2', alpha=0.7))
        if row == n_rows - 1:
            ax_raster.set_xticks(xticks)
            ax_raster.set_xticklabels([f"{b:.2f}" for b in xticks], rotation=0, fontsize=9)
            ax_raster.set_xlabel(f'Time from {align_label} (s)')
        else:
            ax_raster.set_xticklabels([])
        if col_pair == clusters_per_row - 1:
            ax_raster.set_ylabel('Trial')
        if row == 0:
            ax_psth.set_title('PSTH')
            ax_raster.set_title('Raster')
    for row in range(n_rows):
        for i in range(clusters_per_row - 1):
            spacer_col = i * 3 + 2
            axes[row, spacer_col].axis('off')
    plt.tight_layout(h_pad=0.2, w_pad=0.05)
    if save_figure:
        output_dir = 'png_output'
        os.makedirs(output_dir, exist_ok=True)
        fig.savefig(os.path.join(output_dir, f'clusters_grid_sorted_Hz.png'), dpi=150)
        plt.close(fig)
    else:
        plt.show()

# --- Mean PSTH Comparison Function ---
def plot_mean_psth_comparison(session, align_to='Baseline_ON', outcomes_to_compare=['Miss','FA','Hit','abort'], min_trials_with_spikes=20, use_same_clusters=True):
    npx_probes = session['NPX_probes']
    ni_events = session['NI_events']
    trials, outcomes = get_trials_and_outcomes(session)
    spike_times = npx_probes['st']
    cluster_ids = npx_probes['clu']
    window = [-0.5, 1.0]
    bin_size = 0.02
    bins = np.arange(window[0], window[1] + bin_size, bin_size)
    good_clusters = npx_probes.get('cluster_id_KS_good', np.unique(cluster_ids))
    all_psths_by_outcome = {outcome: {} for outcome in outcomes_to_compare}
    clusters_passing = {outcome: set() for outcome in outcomes_to_compare}
    for outcome_of_interest in outcomes_to_compare:
        event_times, trial_indices, align_label, _ = get_event_times(ni_events, align_to, trials, outcomes, outcome_of_interest)
        n_trials_this_outcome = len(event_times)
        for clu in good_clusters:
            spike_times_clu = spike_times[cluster_ids == clu]
            all_counts = []
            for t in event_times:
                aligned = spike_times_clu - t
                mask_spk = (aligned >= window[0]) & (aligned <= window[1])
                counts, _ = np.histogram(aligned[mask_spk], bins=bins)
                all_counts.append(counts)
            all_counts = np.array(all_counts)
            n_trials_with_spikes = np.sum(np.sum(all_counts, axis=1) > 0)
            if n_trials_with_spikes > min_trials_with_spikes:
                # Compute mean firing rate per bin (Hz): mean spike count per bin per trial, divided by bin size
                psth = np.mean(all_counts, axis=0) / bin_size if len(all_counts) > 0 else np.zeros(len(bins)-1)
                all_psths_by_outcome[outcome_of_interest][clu] = psth
                clusters_passing[outcome_of_interest].add(clu)
    if use_same_clusters:
        clusters_to_use = set.intersection(*[clusters_passing[o] for o in outcomes_to_compare])
    else:
        clusters_to_use = set.union(*[clusters_passing[o] for o in outcomes_to_compare])
    mean_psths = {}
    counts_per_outcome = {}
    for outcome in outcomes_to_compare:
        psths = [all_psths_by_outcome[outcome][clu] for clu in clusters_to_use if clu in all_psths_by_outcome[outcome]]
        psths = np.array(psths)
        mean_psths[outcome] = np.mean(psths, axis=0) if len(psths) > 0 else np.zeros(len(bins)-1)
        counts_per_outcome[outcome] = len(psths)
    # Additional: print mean firing rate for each outcome for debugging
    for outcome in outcomes_to_compare:
        mean_rate = np.mean(mean_psths[outcome])
        print(f"Mean firing rate for {outcome}: {mean_rate:.2f} Hz (n={counts_per_outcome[outcome]})")
    plt.figure(figsize=(8, 5))
    colors = ['b', 'r', 'g', 'm', 'c']
    for i, outcome in enumerate(outcomes_to_compare):
        plt.step(bins[:-1], mean_psths[outcome], where='post', label=f"{outcome} (n={counts_per_outcome[outcome]})", color=colors[i % len(colors)])
    plt.axvline(0, color='k', linestyle='--', lw=0.8)
    plt.xlabel(f'Time from {align_label} (s)')
    plt.ylabel('Firing rate (Hz)')
    plt.title(f'Mean PSTH for outcomes aligned to {align_label}\n(Clusters used: {len(clusters_to_use)})')
    plt.legend()
    plt.tight_layout()
    plt.show()

# --- Heatmap Visualization Function ---
def plot_cluster_heatmap(session, align_to='Baseline_ON', outcome_of_interest='Hit', min_trials_with_spikes=20, normalize=True, save_figure=False):
    npx_probes = session['NPX_probes']
    ni_events = session['NI_events']
    trials, outcomes = get_trials_and_outcomes(session)
    spike_times = npx_probes['st']
    cluster_ids = npx_probes['clu']
    event_times, trial_indices, align_label, _ = get_event_times(ni_events, align_to, trials, outcomes, outcome_of_interest)
    good_clusters = npx_probes.get('cluster_id_KS_good', np.unique(cluster_ids))
    window = [-0.5, 1.0]
    bin_size = 0.05
    bins = np.arange(window[0], window[1] + bin_size, bin_size)
    all_psths = []
    cluster_labels = []
    for clu in good_clusters:
        spike_times_clu = spike_times[cluster_ids == clu]
        all_counts = []
        for t in event_times:
            aligned = spike_times_clu - t
            mask = (aligned >= window[0]) & (aligned <= window[1])
            counts, _ = np.histogram(aligned[mask], bins=bins)
            all_counts.append(counts)
        all_counts = np.array(all_counts)
        n_trials_with_spikes = np.sum(np.sum(all_counts, axis=1) > 0)
        if n_trials_with_spikes > min_trials_with_spikes:
            # Convert to Hz: mean spike count per bin per trial, divided by bin size
            mean_psth = np.mean(all_counts, axis=0) / bin_size
            all_psths.append(mean_psth)
            cluster_labels.append(clu)
    if len(all_psths) == 0:
        print("No clusters passed the spike-in-trials filter.")
        return
    data = np.array(all_psths)
    if normalize:
        data = (data - data.mean(axis=1, keepdims=True)) / (data.std(axis=1, keepdims=True) + 1e-8)
        vmin, vmax = -2, 2
        cmap = 'bone'
    else:
        vmin, vmax = np.percentile(data, 1), np.percentile(data, 99)
        cmap = 'bone'
    peak_times = np.argmax(data, axis=1)
    sort_idx = np.argsort(peak_times)
    data = data[sort_idx]
    cluster_labels = np.array(cluster_labels)[sort_idx]
    plt.figure(figsize=(5, min(3, len(cluster_labels) * 0.15)))
    plt.imshow(data, aspect='auto', cmap=cmap, vmin=vmin, vmax=vmax,
               extent=[bins[0], bins[-1], 0, len(cluster_labels)])
    plt.colorbar(label='Firing rate (z-score)' if normalize else 'Firing rate (Hz)')
    plt.xlabel(f'Time from {align_label} (s)')
    plt.ylabel('Cluster')
    renumber_clusters = True
    yticks = np.linspace(0.5, len(cluster_labels)-0.5, min(5, len(cluster_labels)))
    if renumber_clusters:
        yticklabels = [str(int(i)+1) for i in np.linspace(len(cluster_labels)-1,0, len(yticks)).astype(int)]
        plt.ylabel('Cluster (renumbered)')
    else:
        yticklabels = [cluster_labels[int(i)] for i in np.linspace(0, len(cluster_labels)-1, len(yticks)).astype(int)]
        plt.ylabel('Cluster')
    plt.yticks(yticks, yticklabels)
    xticks = np.arange(bins[0], bins[-1] + 0.001, 0.25)
    if 0 not in xticks:
        xticks = np.sort(np.append(xticks, 0))
    xticks = np.unique(np.concatenate(([bins[0]], xticks, [bins[-1]])))
    xticks = np.round(xticks, 8)
    plt.xticks(xticks, [f"{x:.2f}" for x in xticks])
    plt.axvline(0, color='red', linestyle='--', linewidth=1)
    plt.title(f'heatmap around {align_to} for {outcome_of_interest}')
    plt.tight_layout()
    if save_figure:
        output_dir = 'png_output'
        os.makedirs(output_dir, exist_ok=True)
        plt.savefig(os.path.join(output_dir, f'{align_to}_{outcome_of_interest}_clusters_heatmap_sorted_peakHz.png'), dpi=150)
        plt.close()
    else:
        plt.show()

def plot_pfpsth_raster_grid_sorted(session, fast_pulse_thresh=0.25, window=[-0.5, 0.5], bin_size=0.025, clusters_per_row=10, min_fast_pulses=20, min_spikes_per_window=1, save_figure=False):
    npx_probes = session['NPX_probes']
    trials, outcomes = get_trials_and_outcomes(session)
    spike_times = npx_probes['st']
    cluster_ids = npx_probes['clu']
    good_clusters = npx_probes.get('cluster_id_KS_good', np.unique(cluster_ids))
    ni_events = session['NI_events']
    # --- Collect all valid fast pulse times across all trials ---
    all_fast_pulse_times = []
    for trial_idx, trial in enumerate(trials):
        TF_vec_full = np.array(trial['St1TrialVector'])
        TF_vec = TF_vec_full[::3]
        # Get Baseline_ON for this trial
        if 'Baseline_ON' in ni_events:
            baseline_on = ni_events['Baseline_ON']
            if isinstance(baseline_on, dict) and 'rise_t' in baseline_on:
                baseline_on_times = np.array(baseline_on['rise_t']).flatten()
            else:
                baseline_on_times = np.array(baseline_on).flatten()
            t0 = baseline_on_times[trial_idx]
        else:
            continue
        # Get Change_ON for this trial (if available)
        if 'Change_ON' in ni_events:
            change_on = ni_events['Change_ON']
            if isinstance(change_on, dict) and 'rise_t' in change_on:
                change_on_times = np.array(change_on['rise_t']).flatten()
            else:
                change_on_times = np.array(change_on).flatten()
            t_change = change_on_times[trial_idx] if trial_idx < len(change_on_times) else None
        else:
            t_change = None
        # Get outcome and outcome time for this trial
        outcome = trial['trialoutcome']
        reactiontimes = trial.get('reactiontimes', {})
        # For FA/abort, get the outcome time from reactiontimes
        if outcome in ['FA', 'abort']:
            t_outcome = reactiontimes.get(outcome, np.nan)
            if not np.isnan(t_outcome):
                t_outcome = t0 + t_outcome
            else:
                t_outcome = None
        else:
            t_outcome = None
        log2_TF = np.log2(TF_vec + 1e-8)
        fast_pulse_bins = np.where(log2_TF > fast_pulse_thresh)[0]
        fast_pulse_times = fast_pulse_bins * 0.05 + t0
        # Apply filtering conditions
        valid_fast_pulse_times = []
        for fp_time in fast_pulse_times:
            # Condition c: at least 1s after Baseline_ON
            if fp_time < t0 + 1.0:
                continue
            # Condition a: up to 1s before Change_ON (if Change_ON exists)
            if t_change is not None:
                if fp_time > t_change - 1.0:
                    continue
            # Condition b: for FA/abort, up to 2s before outcome time (if no Change_ON)
            elif outcome in ['FA', 'abort'] and t_outcome is not None:
                if fp_time > t_outcome - 2.0:
                    continue
            valid_fast_pulse_times.append(fp_time)
        all_fast_pulse_times.extend(valid_fast_pulse_times)
    all_fast_pulse_times = np.array(all_fast_pulse_times)
    print(f"Total valid fast pulses in session: {len(all_fast_pulse_times)}")
    bins = np.arange(window[0], window[1] + bin_size, bin_size)
    window_length = window[1] - window[0]
    clusters_to_plot = []
    rasters_for_plot = []
    counts_for_plot = []
    mean_rates = []
    for clu in good_clusters:
        spike_times_clu = spike_times[cluster_ids == clu]
        peri_spike_times = []
        for t_fp in all_fast_pulse_times:
            aligned = spike_times_clu - t_fp
            peri_spike_times.append(aligned[(aligned >= window[0]) & (aligned <= window[1])])
        n_fast_pulses_with_spikes = sum(len(s) >= min_spikes_per_window for s in peri_spike_times)
        if n_fast_pulses_with_spikes > min_fast_pulses:
            clusters_to_plot.append(clu)
            rasters_for_plot.append(peri_spike_times)
            all_aligned = np.concatenate(peri_spike_times) if len(peri_spike_times) > 0 else np.array([])
            counts, _ = np.histogram(all_aligned, bins=bins)
            counts_for_plot.append(counts)
            mean_rate = np.sum(counts) / (len(all_fast_pulse_times) * window_length)
            mean_rates.append(mean_rate)
    if len(mean_rates) == 0:
        print("No clusters passed the fast-pulse filter. Adjust your parameters.")
        return
    sort_idx = np.argsort(mean_rates)[::-1]
    clusters_to_plot = [clusters_to_plot[i] for i in sort_idx]
    rasters_for_plot = [rasters_for_plot[i] for i in sort_idx]
    counts_for_plot = [counts_for_plot[i] for i in sort_idx]
    mean_rates = [mean_rates[i] for i in sort_idx]
    n_plot = len(clusters_to_plot)
    n_cols = clusters_per_row * 2 + (clusters_per_row - 1)
    n_rows = int(np.ceil(n_plot / clusters_per_row))
    width_ratios = []
    for i in range(clusters_per_row):
        width_ratios.extend([1, 1])
        if i < clusters_per_row - 1:
            width_ratios.append(0.3)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(2.2 * n_cols, 1.8 * n_rows), sharex='col', gridspec_kw={'width_ratios': width_ratios})
    if n_rows == 1:
        axes = np.expand_dims(axes, axis=0)
    for idx, (clu, peri_spike_times, counts) in enumerate(zip(clusters_to_plot, rasters_for_plot, counts_for_plot)):
        row = idx // clusters_per_row
        col_pair = idx % clusters_per_row
        base_col = col_pair * 3
        ax_psth = axes[row, base_col]
        ax_raster = axes[row, base_col + 1]
        psth = counts / (len(all_fast_pulse_times) * bin_size) if len(all_fast_pulse_times) > 0 else np.zeros(len(bins)-1)
        ax_psth.step(bins[:-1], psth, where='post', color='k')
        ax_psth.axvline(0, color='r', linestyle='--', lw=0.8)
        ax_psth.set_ylim(bottom=0)
        ax_psth.tick_params(axis='y', labelsize=7)
        ax_psth.text(0.98, 0.95, f'Clu {clu}', ha='right', va='top', fontsize=8, transform=ax_psth.transAxes,
                     bbox=dict(facecolor='white', edgecolor='gray', boxstyle='round,pad=0.2', alpha=0.7))
        if row == n_rows - 1:
            xticks = [window[0], 0, window[1]]
            ax_psth.set_xticks(xticks)
            ax_psth.set_xticklabels([f"{b:.2f}" for b in xticks], rotation=0, fontsize=9)
            ax_psth.set_xlabel('Time from fast pulse (s)')
        else:
            ax_psth.set_xticklabels([])
        if col_pair == 0:
            ax_psth.set_ylabel('Firing rate (Hz)')
        n_events_raster = len(peri_spike_times)
        for event, ras in enumerate(peri_spike_times):
            ax_raster.vlines(ras, event + 0.5, event + 1.5, color='k', lw=0.5)
        ax_raster.axvline(0, color='r', linestyle='--', lw=0.8)
        ax_raster.set_ylim(0.5, n_events_raster + 0.5)
        if n_events_raster >= 3:
            event_ticks = [1, n_events_raster // 2 + 1, n_events_raster]
        else:
            event_ticks = list(range(1, n_events_raster + 1))
        ax_raster.set_yticks(event_ticks)
        ax_raster.set_yticklabels([str(t) for t in event_ticks], fontsize=7)
        ax_raster.invert_yaxis()
        ax_raster.text(0.98, 0.95, f'Clu {clu}', ha='right', va='top', fontsize=8, transform=ax_raster.transAxes,
                      bbox=dict(facecolor='white', edgecolor='gray', boxstyle='round,pad=0.2', alpha=0.7))
        if row == n_rows - 1:
            ax_raster.set_xticks(xticks)
            ax_raster.set_xticklabels([f"{b:.2f}" for b in xticks], rotation=0, fontsize=9)
            ax_raster.set_xlabel('Time from fast pulse (s)')
        else:
            ax_raster.set_xticklabels([])
        if col_pair == clusters_per_row - 1:
            ax_raster.set_ylabel('Event')
        if row == 0:
            ax_psth.set_title('PFPSTH')
            ax_raster.set_title('Raster')
    for row in range(n_rows):
        for i in range(clusters_per_row - 1):
            spacer_col = i * 3 + 2
            axes[row, spacer_col].axis('off')
    plt.tight_layout(h_pad=0.2, w_pad=0.05)
    if save_figure:
        output_dir = 'png_output'
        os.makedirs(output_dir, exist_ok=True)
        fig.savefig(os.path.join(output_dir, f'pfpsth_clusters_grid_sorted_Hz.png'), dpi=150)
        plt.close(fig)
    else:
        plt.show()

# --- Fast TF-responsive cluster pfPSTH plotting (no rasters, only PSTH for TF-responsive clusters) ---
def plot_pfpsth_grid_TFresponsive_only(session, fast_pulse_thresh=0.25, window=[-0.5, 0.5], bin_size=0.025, min_fast_pulses=20, min_spikes_per_window=1, tf_response_window=[0,0.3], tf_response_thresh=2.0, save_figure=False):
    import numpy as np
    import matplotlib.pyplot as plt
    npx_probes = session['NPX_probes']
    trials, _ = get_trials_and_outcomes(session)
    spike_times = npx_probes['st']
    cluster_ids = npx_probes['clu']
    good_clusters = npx_probes.get('cluster_id_KS_good', np.unique(cluster_ids))
    ni_events = session['NI_events']
    # --- Collect all valid fast pulse times across all trials ---
    all_fast_pulse_times = []
    for trial_idx, trial in enumerate(trials):
        TF_vec_full = np.array(trial['St1TrialVector'])
        TF_vec = TF_vec_full[::3]
        if 'Baseline_ON' in ni_events:
            baseline_on = ni_events['Baseline_ON']
            if isinstance(baseline_on, dict) and 'rise_t' in baseline_on:
                baseline_on_times = np.array(baseline_on['rise_t']).flatten()
            else:
                baseline_on_times = np.array(baseline_on).flatten()
            t0 = baseline_on_times[trial_idx]
        else:
            continue
        if 'Change_ON' in ni_events:
            change_on = ni_events['Change_ON']
            if isinstance(change_on, dict) and 'rise_t' in change_on:
                change_on_times = np.array(change_on['rise_t']).flatten()
            else:
                change_on_times = np.array(change_on).flatten()
            t_change = change_on_times[trial_idx] if trial_idx < len(change_on_times) else None
        else:
            t_change = None
        outcome = trial['trialoutcome']
        reactiontimes = trial.get('reactiontimes', {})
        if outcome in ['FA', 'abort']:
            t_outcome = reactiontimes.get(outcome, np.nan)
            if not np.isnan(t_outcome):
                t_outcome = t0 + t_outcome
            else:
                t_outcome = None
        else:
            t_outcome = None
        log2_TF = np.log2(TF_vec + 1e-8)
        fast_pulse_bins = np.where(log2_TF > fast_pulse_thresh)[0]
        fast_pulse_times = fast_pulse_bins * 0.05 + t0
        valid_fast_pulse_times = []
        for fp_time in fast_pulse_times:
            if fp_time < t0 + 1.0:
                continue
            if t_change is not None:
                if fp_time > t_change - 1.0:
                    continue
            elif outcome in ['FA', 'abort'] and t_outcome is not None:
                if fp_time > t_outcome - 2.0:
                    continue
            valid_fast_pulse_times.append(fp_time)
        all_fast_pulse_times.extend(valid_fast_pulse_times)
    all_fast_pulse_times = np.array(all_fast_pulse_times)
    print(f"Total valid fast pulses in session: {len(all_fast_pulse_times)}")
    bins = np.arange(window[0], window[1] + bin_size, bin_size)
    window_length = window[1] - window[0]
    tf_responsive_clusters = []
    psths_for_plot = []
    for clu in good_clusters:
        spike_times_clu = spike_times[cluster_ids == clu]
        peri_spike_times = []
        for t_fp in all_fast_pulse_times:
            aligned = spike_times_clu - t_fp
            peri_spike_times.append(aligned[(aligned >= window[0]) & (aligned <= window[1])])
        n_fast_pulses_with_spikes = sum(len(s) >= min_spikes_per_window for s in peri_spike_times)
        if n_fast_pulses_with_spikes > min_fast_pulses:
            # Compute PSTH
            all_aligned = np.concatenate(peri_spike_times) if len(peri_spike_times) > 0 else np.array([])
            counts, _ = np.histogram(all_aligned, bins=bins)
            psth = counts / (len(all_fast_pulse_times) * bin_size) if len(all_fast_pulse_times) > 0 else np.zeros(len(bins)-1)
            # TF-responsiveness: mean firing in tf_response_window vs baseline
            tf_mask = (bins[:-1] >= tf_response_window[0]) & (bins[:-1] < tf_response_window[1])
            base_mask = (bins[:-1] < 0)
            tf_rate = np.mean(psth[tf_mask])
            base_rate = np.mean(psth[base_mask])
            if (tf_rate - base_rate) > tf_response_thresh:
                tf_responsive_clusters.append(clu)
                psths_for_plot.append(psth)
    n_plot = len(tf_responsive_clusters)
    print(f"TF-responsive clusters: {n_plot}")
    if n_plot == 0:
        print("No TF-responsive clusters found.")
        return
    # Plot grid of PSTHs only
    n_cols = 5
    n_rows = int(np.ceil(n_plot / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(2.2 * n_cols, 1.8 * n_rows), sharex=True, sharey=True)
    axes = axes.flatten()
    for idx, (clu, psth) in enumerate(zip(tf_responsive_clusters, psths_for_plot)):
        ax = axes[idx]
        ax.step(bins[:-1], psth, where='post', color='k')
        ax.axvline(0, color='r', linestyle='--', lw=0.8)
        ax.set_ylim(bottom=0)
        ax.set_title(f'Clu {clu}', fontsize=9)
        if idx % n_cols == 0:
            ax.set_ylabel('Firing rate (Hz)')
        if idx // n_cols == n_rows - 1:
            ax.set_xlabel('Time from fast pulse (s)')
    for ax in axes[n_plot:]:
        ax.axis('off')
    plt.tight_layout(h_pad=0.2, w_pad=0.05)
    if save_figure:
        output_dir = 'png_output'
        os.makedirs(output_dir, exist_ok=True)
        fig.savefig(os.path.join(output_dir, f'pfpsth_TFresponsive_grid_Hz.png'), dpi=150)
        plt.close(fig)
    else:
        plt.show()

# --- PSTH Comparison Aligned to Reaction Time for Each Outcome ---
def get_true_reaction_time(trial, ni_events, trial_idx, shift_fa_hit_ms=200.0):
    outcome = trial['trialoutcome']
    reactiontimes = trial.get('reactiontimes', {})
    # Get Baseline_ON time
    if 'Baseline_ON' in ni_events:
        baseline_on = ni_events['Baseline_ON']
        if isinstance(baseline_on, dict) and 'rise_t' in baseline_on:
            baseline_on_times = np.array(baseline_on['rise_t']).flatten()
        else:
            baseline_on_times = np.array(baseline_on).flatten()
        t0 = baseline_on_times[trial_idx]
    else:
        return None
    # Get Change_ON time (if available)
    if 'Change_ON' in ni_events:
        change_on = ni_events['Change_ON']
        if isinstance(change_on, dict) and 'rise_t' in change_on:
            change_on_times = np.array(change_on['rise_t']).flatten()
        else:
            change_on_times = np.array(change_on).flatten()
        t_change = change_on_times[trial_idx] if trial_idx < len(change_on_times) else None
    else:
        t_change = None
    # Compute true reaction time
    shift = 0.0
    if outcome in ['FA', 'Hit']:
        shift = shift_fa_hit_ms / 1000.0  # convert ms to s
    if outcome == 'Hit':
        rt = reactiontimes.get('RT', np.nan)
        if not np.isnan(rt) and t_change is not None:
            return t_change + rt - shift
        else:
            return None
    elif outcome == 'Miss':
        rt = reactiontimes.get('Miss', np.nan)
        if not np.isnan(rt) and t_change is not None:
            return t_change + rt
        else:
            return None
    elif outcome in ['FA', 'abort']:
        rt = reactiontimes.get(outcome, np.nan)
        if not np.isnan(rt):
            return t0 + rt - shift if outcome == 'FA' else t0 + rt
        else:
            return None
    else:
        return None

def plot_psth_aligned_to_reaction_time(session, outcomes_to_compare=['Hit','Miss','FA','abort'], window=[-0.5, 0.5], bin_size=0.02, min_trials_with_spikes=10, use_same_clusters=True, shift_fa_hit_ms=200.0):
    npx_probes = session['NPX_probes']
    ni_events = session['NI_events']
    trials, outcomes = get_trials_and_outcomes(session)
    spike_times = npx_probes['st']
    cluster_ids = npx_probes['clu']
    good_clusters = npx_probes.get('cluster_id_KS_good', np.unique(cluster_ids))
    bins = np.arange(window[0], window[1] + bin_size, bin_size)
    all_psths_by_outcome = {outcome: {} for outcome in outcomes_to_compare}
    clusters_passing = {outcome: set() for outcome in outcomes_to_compare}
    for outcome_of_interest in outcomes_to_compare:
        trial_indices = np.where(outcomes == outcome_of_interest)[0]
        event_times = []
        for idx in trial_indices:
            t = get_true_reaction_time(trials[idx], ni_events, idx, shift_fa_hit_ms=shift_fa_hit_ms)
            if t is not None:
                event_times.append(t)
        event_times = np.array(event_times)
        for clu in good_clusters:
            spike_times_clu = spike_times[cluster_ids == clu]
            all_counts = []
            for t in event_times:
                aligned = spike_times_clu - t
                mask_spk = (aligned >= window[0]) & (aligned <= window[1])
                counts, _ = np.histogram(aligned[mask_spk], bins=bins)
                all_counts.append(counts)
            all_counts = np.array(all_counts)
            if all_counts.ndim == 1 or all_counts.shape[0] == 0:
                continue
            n_trials_with_spikes = np.sum(np.sum(all_counts, axis=1) > 0)
            if n_trials_with_spikes > min_trials_with_spikes:
                psth = np.mean(all_counts, axis=0) / bin_size if len(all_counts) > 0 else np.zeros(len(bins)-1)
                all_psths_by_outcome[outcome_of_interest][clu] = psth
                clusters_passing[outcome_of_interest].add(clu)
    if use_same_clusters:
        clusters_to_use = set.intersection(*[clusters_passing[o] for o in outcomes_to_compare])
    else:
        clusters_to_use = set.union(*[clusters_passing[o] for o in outcomes_to_compare])
    mean_psths = {}
    counts_per_outcome = {}
    for outcome in outcomes_to_compare:
        psths = [all_psths_by_outcome[outcome][clu] for clu in clusters_to_use if clu in all_psths_by_outcome[outcome]]
        psths = np.array(psths)
        mean_psths[outcome] = np.mean(psths, axis=0) if len(psths) > 0 else np.zeros(len(bins)-1)
        counts_per_outcome[outcome] = len(psths)
    plt.figure(figsize=(8, 5))
    colors = ['b', 'r', 'g', 'm', 'c']
    for i, outcome in enumerate(outcomes_to_compare):
        plt.step(bins[:-1], mean_psths[outcome], where='post', label=f"{outcome} (n={counts_per_outcome[outcome]})", color=colors[i % len(colors)])
    plt.axvline(0, color='k', linestyle='--', lw=0.8)
    plt.xlabel('Time from reaction time (s)')
    plt.ylabel('Firing rate (Hz)')
    plt.title('Mean PSTH for outcomes aligned to true reaction time\n(Clusters used: %d)' % len(clusters_to_use))
    plt.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
# Example usage:
mat_path = 'BG_031_260325.mat'
data = load_mat_file(mat_path)
data_dict1 = mat_struct_to_dict(data)
data_dict = get_session_dict(data_dict1)
session = get_drilled_down(data_dict, 'BG_031', 'BG_031_260325')

In [3]:
from tf_responsive_unit_screening_v2 import find_tf_responsive_units_activity, plot_tf_responsive_psths_activity
tf_responsive_clusters, z_scores, all_fast_pulse_times = find_tf_responsive_units_activity(session)
# if len(tf_units) > 0:
#     plot_tf_responsive_psths(session, tf_units, fast_pulse_times)

Total valid fast pulses in session: 9424
Total clusters to check: 185
Processing cluster 1/185 (clu=1)... Elapsed: 1.3s


ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

In [6]:
plot_tf_responsive_psths_activity(session, tf_responsive_clusters, all_fast_pulse_times)

KeyboardInterrupt: 

In [ ]:
fast_pulse_times

array([   70.27272968,    70.42272968,    71.07272968, ...,
       11111.16620851, 11120.54243855, 11141.24288789])

In [ ]:

# Sorted PSTH/raster grid
# plot_psth_raster_grid_sorted(session, align_to='Change_ON', outcome_of_interest='Hit', clusters_per_row=10, min_trials_with_spikes=20)

# Mean PSTH comparison
plot_mean_psth_comparison(session, align_to='Change_ON', outcomes_to_compare=['Hit','Miss'], min_trials_with_spikes=20, use_same_clusters=True)

# Cluster heatmap
plot_cluster_heatmap(session, align_to='Baseline_ON', outcome_of_interest='Hit', min_trials_with_spikes=20, normalize=True)

# Plot PSTH aligned to true reaction time for each outcome
plot_psth_aligned_to_reaction_time(session, outcomes_to_compare=['Hit','Miss','FA','abort'], window=[-0.5,0.5], bin_size=0.02, min_trials_with_spikes=10, use_same_clusters=True , shift_fa_hit_ms=200.0)

In [ ]:
# Screen for trials where Change_ON is NaN (missing) for quick diagnostics

trials, outcomes = get_trials_and_outcomes(session)
ni_events = session['NI_events']

# Get Change_ON times for all trials
if 'Change_ON' in ni_events:
    change_on = ni_events['Change_ON']
    if isinstance(change_on, dict) and 'rise_t' in change_on:
        change_on_times = np.array(change_on['rise_t']).flatten()
    else:
        change_on_times = np.array(change_on).flatten()
else:
    change_on_times = None

nan_indices = []
if change_on_times is not None:
    for idx, t in enumerate(change_on_times):
        if np.isnan(t):
            nan_indices.append(idx)
    print(f"Number of trials with Change_ON as NaN: {len(nan_indices)}")
    if nan_indices:
        print("Indices of trials with NaN Change_ON:", nan_indices)
        print("Outcomes for these trials:", [outcomes[i] for i in nan_indices])
else:
    print("No Change_ON event found in NI_events.")


Number of trials with Change_ON as NaN: 571
Indices of trials with NaN Change_ON: [0, 1, 2, 3, 4, 6, 7, 9, 12, 13, 14, 15, 19, 20, 21, 22, 23, 24, 26, 27, 28, 29, 30, 35, 36, 37, 38, 41, 43, 44, 45, 46, 47, 49, 50, 51, 55, 56, 58, 59, 61, 62, 64, 66, 67, 69, 71, 73, 76, 78, 80, 81, 85, 86, 88, 90, 92, 94, 95, 97, 98, 99, 101, 103, 104, 105, 106, 108, 109, 113, 114, 118, 119, 120, 121, 122, 123, 126, 127, 128, 129, 130, 131, 132, 133, 137, 138, 140, 141, 142, 143, 144, 145, 146, 155, 157, 159, 162, 163, 169, 170, 171, 172, 177, 178, 184, 189, 190, 195, 197, 200, 203, 204, 206, 207, 208, 209, 210, 211, 212, 213, 215, 216, 217, 224, 227, 229, 230, 233, 239, 240, 241, 247, 258, 263, 264, 265, 266, 269, 272, 273, 279, 282, 284, 285, 287, 291, 292, 293, 294, 295, 297, 299, 301, 303, 304, 305, 306, 307, 308, 311, 312, 313, 315, 316, 317, 318, 320, 323, 324, 325, 326, 327, 328, 330, 331, 332, 333, 334, 336, 338, 339, 340, 341, 342, 343, 345, 346, 347, 348, 349, 350, 352, 355, 357, 358, 360, 36

In [ ]:

# --- Diagnostic: PSTH bin edges and alignment to reaction time (0) ---
def diagnostic_plot_psth_bin_alignment(session, outcomes_to_compare=['Hit','Miss','FA','abort'], window=[-0.5, 0.5], bin_size=0.02, min_trials_with_spikes=10, use_same_clusters=True, shift_fa_hit_ms=0.0):
    import matplotlib.pyplot as plt
    npx_probes = session['NPX_probes']
    ni_events = session['NI_events']
    trials, outcomes = get_trials_and_outcomes(session)
    spike_times = npx_probes['st']
    cluster_ids = npx_probes['clu']
    good_clusters = npx_probes.get('cluster_id_KS_good', np.unique(cluster_ids))
    bins = np.arange(window[0], window[1] + bin_size, bin_size)
    all_psths_by_outcome = {outcome: {} for outcome in outcomes_to_compare}
    clusters_passing = {outcome: set() for outcome in outcomes_to_compare}
    for outcome_of_interest in outcomes_to_compare:
        trial_indices = np.where(outcomes == outcome_of_interest)[0]
        event_times = []
        for idx in trial_indices:
            t = get_true_reaction_time(trials[idx], ni_events, idx, shift_fa_hit_ms=shift_fa_hit_ms)
            if t is not None:
                event_times.append(t)
        event_times = np.array(event_times)
        for clu in good_clusters:
            spike_times_clu = spike_times[cluster_ids == clu]
            all_counts = []
            for t in event_times:
                aligned = spike_times_clu - t
                mask_spk = (aligned >= window[0]) & (aligned <= window[1])
                counts, _ = np.histogram(aligned[mask_spk], bins=bins)
                all_counts.append(counts)
            all_counts = np.array(all_counts)
            if all_counts.ndim == 1 or all_counts.shape[0] == 0:
                continue
            n_trials_with_spikes = np.sum(np.sum(all_counts, axis=1) > 0)
            if n_trials_with_spikes > min_trials_with_spikes:
                psth = np.mean(all_counts, axis=0) / bin_size if len(all_counts) > 0 else np.zeros(len(bins)-1)
                all_psths_by_outcome[outcome_of_interest][clu] = psth
                clusters_passing[outcome_of_interest].add(clu)
    if use_same_clusters:
        clusters_to_use = set.intersection(*[clusters_passing[o] for o in outcomes_to_compare])
    else:
        clusters_to_use = set.union(*[clusters_passing[o] for o in outcomes_to_compare])
    mean_psths = {}
    counts_per_outcome = {}
    for outcome in outcomes_to_compare:
        psths = [all_psths_by_outcome[outcome][clu] for clu in clusters_to_use if clu in all_psths_by_outcome[outcome]]
        psths = np.array(psths)
        mean_psths[outcome] = np.mean(psths, axis=0) if len(psths) > 0 else np.zeros(len(bins)-1)
        counts_per_outcome[outcome] = len(psths)
    plt.figure(figsize=(8, 5))
    colors = ['b', 'r', 'g', 'm', 'c']
    for i, outcome in enumerate(outcomes_to_compare):
        plt.step(bins[:-1], mean_psths[outcome], where='post', label=f"{outcome} (n={counts_per_outcome[outcome]})", color=colors[i % len(colors)])
    plt.axvline(0, color='k', linestyle='--', lw=1.2, label='Reaction time (0)')
    for b in bins:
        plt.axvline(b, color='gray', linestyle=':', alpha=0.2)
    plt.xlabel('Time from reaction time (s)')
    plt.ylabel('Firing rate (Hz)')
    plt.title('Diagnostic: PSTH bin edges and alignment to reaction time (0)')
    plt.legend()
    plt.tight_layout()
    plt.show()